# LeetCode 959 — Regions Cut By Slashes bằng Python

Link đề: https://leetcode.com/problems/regions-cut-by-slashes/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Depth-First Search (DFS), Breadth-First Search (BFS), Union Find (tập hợp rời nhau), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        ...
```

Ghi chú: lưới `n x n` gồm các ô `1 x 1`, mỗi ô chứa dấu `/`, dấu `\` hoặc khoảng trắng; các dấu gạch chéo chia lưới thành các **vùng**. Điểm khó: một ô có dấu gạch chéo bị **chia đôi theo đường chéo**, nên không thể coi mỗi ô là một "đỉnh". Cách quen thuộc: **chia mỗi ô thành 4 tam giác** (trên, phải, dưới, trái) rồi **gộp** các tam giác thông nhau (trong ô theo dấu, và giữa hai ô kề nhau qua cạnh chung); số vùng là số **thành phần liên thông**. Lưu ý: dấu gạch chéo ngược `\` trong đầu vào được viết là `"\\"` trong chuỗi Python (một ký tự).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số vùng mà các dấu `/` và `\` chia hình vuông lớn thành.
2. Nắm các công cụ Python cần thiết: chuỗi có ký tự gạch chéo ngược (`"\\"`), chỉ số phẳng `4 * (i * n + j) + k`, cấu trúc Union Find (tìm đại diện có nén đường, gộp), DFS/BFS trên đồ thị ngầm, và phóng to mỗi ô thành lưới `3 x 3` điểm ảnh.
3. Hiểu nguyên lý: **tách mỗi ô thành 4 tam giác** để mô hình hóa việc dấu gạch chéo chỉ **chặn đường** giữa các tam giác ở hai phía của nó; số vùng là số thành phần liên thông của đồ thị tam giác.
4. Đi qua 4 cách: phóng to `3 x 3` rồi tô loang (brute, oracle), Union Find trên 4 tam giác mỗi ô (cách nộp chính), DFS trên đồ thị tam giác, và Union Find trên **các điểm lưới** (đếm chu trình); đối chiếu giữa các cách.
5. Tránh các lỗi kinh điển: không gộp các tam giác giữa hai ô kề, gộp sai tam giác cho `/` và `\`, coi ô trống là bốn mảnh rời, và quên nối các điểm trên viền ở cách điểm lưới.

## 2. Đề bài & ví dụ

Một lưới `n x n` gồm các ô `1 x 1`; mỗi ô chứa `'/'`, `'\'` hoặc khoảng trắng `' '`. Các ký tự này chia lưới thành các vùng liên tục.

Cho `grid` là mảng chuỗi, trả về **số vùng**.

Lưu ý: ký tự `\` được viết dạng `"\\"` trong chuỗi.

**Ví dụ 1:** `grid = [" /","/ "]` -> `2`.

**Ví dụ 2:** `grid = [" /","  "]` -> `1`.

**Ví dụ 3:** `grid = ["/\\","\\/"]` -> `5` (`"/\\"` là `/\`, `"\\/"` là `\/`: hình thoi ở giữa và bốn tam giác ở bốn góc).

**Ràng buộc:**

- `n == grid.length == grid[i].length`
- `1 <= n <= 30`
- `grid[i][j]` là `'/'`, `'\'` hoặc `' '`.

In [1]:
from typing import List, Tuple
from collections import deque
import random, time

examples = [
    ([" /", "/ "], 2),
    ([" /", "  "], 1),
    (["/\\", "\\/"], 5),
]
for g, expected in examples:
    print(f"grid={g} (hiển thị: {[row for row in g]}) kỳ vọng={expected}; độ dài mỗi chuỗi {[len(r) for r in g]}")

grid=[' /', '/ '] (hiển thị: [' /', '/ ']) kỳ vọng=2; độ dài mỗi chuỗi [2, 2]
grid=[' /', '  '] (hiển thị: [' /', '  ']) kỳ vọng=1; độ dài mỗi chuỗi [2, 2]
grid=['/\\', '\\/'] (hiển thị: ['/\\', '\\/']) kỳ vọng=5; độ dài mỗi chuỗi [2, 2]


## 3. Kiến thức nền

### 3.1. Ký tự gạch chéo ngược trong Python

Trong mã Python, ký tự `\` phải viết `"\\"` (hoặc `'\\'`); chuỗi `"/\\"` có **hai** ký tự `/` và `\`. Khi LeetCode đưa đầu vào, mỗi `\\` đã là một ký tự.

In [2]:
s = "/\\"
print(len(s), list(s), s[1] == "\\")

2 ['/', '\\'] True


### 3.2. Bốn tam giác trong mỗi ô

Mỗi ô chia bằng **hai đường chéo** thành 4 tam giác: **0 = trên**, **1 = phải**, **2 = dưới**, **3 = trái**. Chỉ số phẳng của tam giác `k` của ô `(i, j)` là `4 * (i * n + j) + k`.

- Ô trống `' '`: không có chướng ngại, cả 4 tam giác thông nhau.
- Dấu `/` (từ góc dưới trái lên góc trên phải): ngăn cách {trên, trái} với {dưới, phải}: nối `0-3` và `1-2`.
- Dấu `\` (từ góc trên trái xuống góc dưới phải): ngăn cách {trên, phải} với {dưới, trái}: nối `0-1` và `2-3`.
- Giữa hai ô kề: tam giác **dưới** của `(i, j)` nối **trên** của `(i + 1, j)`; tam giác **phải** của `(i, j)` nối **trái** của `(i, j + 1)`.

In [3]:
def piece(n: int, i: int, j: int, k: int) -> int:
    return 4 * (i * n + j) + k

TOP, RIGHT, BOTTOM, LEFT = 0, 1, 2, 3
n = 2
print("ô (1, 0): các tam giác", [piece(n, 1, 0, k) for k in range(4)])
print("nối dưới của (0, 0) =", piece(n, 0, 0, BOTTOM), "với trên của (1, 0) =", piece(n, 1, 0, TOP))

ô (1, 0): các tam giác [8, 9, 10, 11]
nối dưới của (0, 0) = 2 với trên của (1, 0) = 8


### 3.3. Union Find (tập hợp rời nhau)

`find(x)` trả về đại diện của tập chứa `x` (nén đường); `union(a, b)` gộp hai tập, và nếu hai phần tử **đã cùng tập** thì không làm gì. Bắt đầu với `4 * n * n` tập, mỗi lần gộp **thành công** giảm số thành phần đi `1`; kết quả cuối là số vùng.

In [4]:
class DSU:
    def __init__(self, size: int):
        self.parent = list(range(size))
        self.components = size

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        self.parent[ra] = rb
        self.components -= 1
        return True

d = DSU(5)
print(d.union(0, 1), d.union(1, 2), d.union(0, 2), d.components)

True True False 3


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `[" /","/ "]`, `n = 2`; ô `(0,0)` trống, `(0,1)` là `/`, `(1,0)` là `/`, `(1,1)` trống. Có `16` tam giác.

- Mỗi ô trống gộp 4 tam giác thành 1 mảnh; mỗi ô `/` gộp thành 2 mảnh. Sau gộp trong ô: `1 + 2 + 2 + 1 = 6` mảnh.
- Gộp giữa các ô: dưới của `(0,0)` với trên của `(1,0)`: mảnh của ô trống nối mảnh {trên, trái} của ô `/` phía dưới. Tương tự các cạnh khác.
- Kết quả: hai vùng: hai đường chéo `/` song song ở hai ô `(0,1)` và `(1,0)` chia hình vuông thành hai phần (góc trên trái và góc dưới phải), vì hai đường nối vào nhau ở tâm lưới tạo thành một đường chéo liên tục từ góc dưới trái tới góc trên phải của lưới; vậy đáp án `2`.

## 5. Cách 1 — Brute Force: phóng to `3 x 3` rồi tô loang

**Ý tưởng.** Mỗi ô thành khối `3 x 3` điểm ảnh: dấu `/` tô các điểm `(0,2), (1,1), (2,0)`; dấu `\` tô `(0,0), (1,1), (2,2)`; ô trống không tô. Các điểm tô là "tường". Đếm số **thành phần liên thông 4 hướng** của các điểm **không** tô (loang bằng ngăn xếp). Tường chéo là chuỗi điểm chỉ kề nhau theo đường chéo, nên **chặn** loang 4 hướng; kích thước `3 x 3` đủ để mỗi vùng thật (kể cả một tam giác góc) có ít nhất một điểm ảnh. Đây là oracle theo hình học, khác hẳn Union Find.

In [5]:
def regions_upscale(grid: List[str]) -> int:
    n = len(grid)
    N = 3 * n
    wall = [[False] * N for _ in range(N)]
    for i, row in enumerate(grid):
        for j, ch in enumerate(row):
            if ch == "/":
                for k in range(3):
                    wall[3 * i + k][3 * j + 2 - k] = True
            elif ch == "\\":
                for k in range(3):
                    wall[3 * i + k][3 * j + k] = True
    seen = [[False] * N for _ in range(N)]
    count = 0
    for a in range(N):
        for b in range(N):
            if wall[a][b] or seen[a][b]:
                continue
            count += 1
            stack = [(a, b)]
            seen[a][b] = True
            while stack:
                x, y = stack.pop()
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < N and 0 <= ny < N and not wall[nx][ny] and not seen[nx][ny]:
                        seen[nx][ny] = True
                        stack.append((nx, ny))
    return count

for g, expected in examples:
    got = regions_upscale(g)
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[' /', '/ '] -> 2 OK
[' /', '  '] -> 1 OK
['/\\', '\\/'] -> 5 OK


**Độ phức tạp và nút thắt.** `O(9 n^2)` điểm ảnh, mỗi điểm ảnh xử lý `O(1)`: thực chất vẫn tuyến tính theo kích thước lưới. Nhưng nó dùng bộ nhớ gấp `9` lần và là cách "giải bằng hình ảnh"; ta cần một mô hình đồ thị gọn hơn và tổng quát hơn (ví dụ để làm việc với hình dạng khác ngoài `/` và `\`). Xem thời gian:

In [6]:
def random_grid(rng, n, p_slash=0.5):
    chars = []
    out = []
    for _ in range(n):
        row = ""
        for _ in range(n):
            r = rng.random()
            row += " " if r > p_slash else ("/" if rng.random() < 0.5 else "\\")
        out.append(row)
    return out

rng = random.Random(959)
for n in (5, 10, 20, 30):
    g = random_grid(rng, n)
    t0 = time.perf_counter()
    r = regions_upscale(g)
    print(f"n={n:2}: {r:3} vùng, {1000 * (time.perf_counter() - t0):7.2f} ms")

n= 5:   4 vùng,    0.18 ms
n=10:   4 vùng,    0.72 ms
n=20:   9 vùng,    2.39 ms
n=30:  21 vùng,    5.03 ms


Nút thắt: phóng to làm tăng kích thước dữ liệu và chỉ giải bài toán đặc thù này; mô hình **bốn tam giác** cho đồ thị nhỏ hơn (`4 n^2` đỉnh) và áp dụng Union Find hiệu quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Tách ô thành 4 tam giác.** Một dấu gạch chéo chỉ ngăn cách các tam giác **ở hai phía** của nó; các tam giác cùng phía thông nhau, và hai tam giác của hai ô kề nhau chung một cạnh luôn thông nhau (cạnh ô không bao giờ bị chặn). Do đó số vùng = số thành phần liên thông của đồ thị gồm `4 n^2` tam giác với các cạnh nói trên.

| Cách | Ý tưởng |
|---|---|
| 2 | Union Find trên các tam giác (nộp) |
| 3 | DFS (hoặc BFS) trên đồ thị tam giác |
| 4 | Union Find trên **điểm lưới**: mỗi dấu là một cạnh; cạnh nối hai điểm đã liên thông tạo thêm một vùng |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Union Find trên các tam giác (cách nộp chính)

**Giải thích từng đoạn.**

- `dsu = DSU(4 * n * n)`; `count` ban đầu là `4 n^2`.
- Trong ô: ô trống gộp `0-1`, `1-2`, `2-3`; ô `/` gộp `0-3` và `1-2`; ô `\` gộp `0-1` và `2-3`.
- Giữa ô: `BOTTOM(i, j)` với `TOP(i + 1, j)`; `RIGHT(i, j)` với `LEFT(i, j + 1)`.
- Kết quả `dsu.components`.

In [7]:
def regions_dsu(grid: List[str]) -> int:
    n = len(grid)
    dsu = DSU(4 * n * n)
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            ch = grid[i][j]
            if ch == " ":
                dsu.union(b, b + 1)
                dsu.union(b + 1, b + 2)
                dsu.union(b + 2, b + 3)
            elif ch == "/":
                dsu.union(b, b + 3)
                dsu.union(b + 1, b + 2)
            else:
                dsu.union(b, b + 1)
                dsu.union(b + 2, b + 3)
            if i + 1 < n:
                dsu.union(b + 2, 4 * ((i + 1) * n + j))
            if j + 1 < n:
                dsu.union(b + 1, 4 * (i * n + j + 1) + 3)
    return dsu.components

for g, expected in examples:
    got = regions_dsu(g)
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[' /', '/ '] -> 2 OK
[' /', '  '] -> 1 OK
['/\\', '\\/'] -> 5 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        n = len(grid)
        parent = list(range(4 * n * n))
        count = 4 * n * n

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a: int, b: int) -> None:
            nonlocal count
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                count -= 1

        for i in range(n):
            for j in range(n):
                b = 4 * (i * n + j)
                ch = grid[i][j]
                if ch == " ":
                    union(b, b + 1)
                    union(b + 1, b + 2)
                    union(b + 2, b + 3)
                elif ch == "/":
                    union(b, b + 3)
                    union(b + 1, b + 2)
                else:
                    union(b, b + 1)
                    union(b + 2, b + 3)
                if i + 1 < n:
                    union(b + 2, 4 * ((i + 1) * n + j))
                if j + 1 < n:
                    union(b + 1, 4 * (i * n + j + 1) + 3)
        return count

sol_dsu = Solution().regionsBySlashes
assert [sol_dsu(list(g)) for g, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n^2 * alpha(n^2))` thời gian (`alpha` là hàm Ackermann ngược, coi như hằng), `O(n^2)` bộ nhớ.

---

### 7.2. Cách 3 — DFS trên đồ thị tam giác

**Ý tưởng khác.** Không cần Union Find: dựng **danh sách kề** của `4 n^2` tam giác với đúng các cạnh như Cách 2, rồi đếm thành phần liên thông bằng DFS (ngăn xếp tường minh).

**Giải thích từng đoạn.** `edges(a, b)` thêm cạnh hai chiều; sau đó duyệt `x` từ `0..4n^2 - 1`: nếu chưa thăm thì `count += 1` và DFS đánh dấu cả thành phần.

In [9]:
def regions_dfs(grid: List[str]) -> int:
    n = len(grid)
    size = 4 * n * n
    adj = [[] for _ in range(size)]

    def link(a: int, b: int) -> None:
        adj[a].append(b)
        adj[b].append(a)

    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            ch = grid[i][j]
            if ch == " ":
                link(b, b + 1); link(b + 1, b + 2); link(b + 2, b + 3)
            elif ch == "/":
                link(b, b + 3); link(b + 1, b + 2)
            else:
                link(b, b + 1); link(b + 2, b + 3)
            if i + 1 < n:
                link(b + 2, 4 * ((i + 1) * n + j))
            if j + 1 < n:
                link(b + 1, 4 * (i * n + j + 1) + 3)
    seen = [False] * size
    count = 0
    for s in range(size):
        if seen[s]:
            continue
        count += 1
        seen[s] = True
        stack = [s]
        while stack:
            x = stack.pop()
            for y in adj[x]:
                if not seen[y]:
                    seen[y] = True
                    stack.append(y)
    return count

for g, expected in examples:
    got = regions_dfs(g)
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[' /', '/ '] -> 2 OK
[' /', '  '] -> 1 OK
['/\\', '\\/'] -> 5 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        n = len(grid)
        size = 4 * n * n
        adj = [[] for _ in range(size)]

        def link(a: int, b: int) -> None:
            adj[a].append(b)
            adj[b].append(a)

        for i in range(n):
            for j in range(n):
                b = 4 * (i * n + j)
                ch = grid[i][j]
                if ch == " ":
                    link(b, b + 1); link(b + 1, b + 2); link(b + 2, b + 3)
                elif ch == "/":
                    link(b, b + 3); link(b + 1, b + 2)
                else:
                    link(b, b + 1); link(b + 2, b + 3)
                if i + 1 < n:
                    link(b + 2, 4 * ((i + 1) * n + j))
                if j + 1 < n:
                    link(b + 1, 4 * (i * n + j + 1) + 3)
        seen = [False] * size
        count = 0
        for s in range(size):
            if seen[s]:
                continue
            count += 1
            seen[s] = True
            stack = [s]
            while stack:
                x = stack.pop()
                for y in adj[x]:
                    if not seen[y]:
                        seen[y] = True
                        stack.append(y)
        return count

sol_dfs = Solution().regionsBySlashes
assert [sol_dfs(list(g)) for g, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n^2)` thời gian và bộ nhớ (đồ thị `4 n^2` đỉnh, mỗi đỉnh tối đa `4` cạnh).

---

### 7.3. Cách 4 — Union Find trên điểm lưới (đếm chu trình)

**Ý tưởng khác.** Xem mỗi dấu là một **cạnh** nối hai **điểm lưới** (góc ô): ô `(i, j)` có bốn góc `(i, j)`, `(i, j + 1)`, `(i + 1, j)`, `(i + 1, j + 1)` (tọa độ điểm theo hàng, cột); `\` nối `(i, j)` với `(i + 1, j + 1)`, `/` nối `(i, j + 1)` với `(i + 1, j)`. Khung hình vuông ngoài cùng cũng là một "tường": mọi điểm trên **viền** coi như đã liên thông với nhau (một thành phần). Khởi đầu có `1` vùng (cả hình vuông). Thêm lần lượt từng cạnh gạch chéo: nếu hai đầu **đã cùng thành phần** thì cạnh này khép một **chu trình** và tách thêm **một vùng** mới (`regions += 1`); ngược lại gộp hai thành phần.

**Giải thích từng đoạn.** Đây là công thức Euler cho đồ thị phẳng: mỗi cạnh khép chu trình tăng số mặt thêm `1`, mỗi cạnh nối hai thành phần khác nhau thì không đổi số mặt. Điểm viền được nối trước để biểu diễn khung.

In [11]:
def regions_points(grid: List[str]) -> int:
    n = len(grid)
    m = n + 1
    dsu = DSU(m * m)
    pid = lambda r, c: r * m + c
    border = [pid(0, c) for c in range(m)] + [pid(m - 1, c) for c in range(m)] + \
             [pid(r, 0) for r in range(m)] + [pid(r, m - 1) for r in range(m)]
    for a, b in zip(border, border[1:]):
        dsu.union(a, b)
    regions = 1
    for i in range(n):
        for j in range(n):
            ch = grid[i][j]
            if ch == " ":
                continue
            if ch == "\\":
                a, b = pid(i, j), pid(i + 1, j + 1)
            else:
                a, b = pid(i, j + 1), pid(i + 1, j)
            if not dsu.union(a, b):
                regions += 1
    return regions

for g, expected in examples:
    got = regions_points(g)
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[' /', '/ '] -> 2 OK
[' /', '  '] -> 1 OK
['/\\', '\\/'] -> 5 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        n = len(grid)
        m = n + 1
        parent = list(range(m * m))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a: int, b: int) -> bool:
            ra, rb = find(a), find(b)
            if ra == rb:
                return False
            parent[ra] = rb
            return True

        for r in range(m):
            for c in range(m):
                if r == 0 or c == 0 or r == m - 1 or c == m - 1:
                    union(r * m + c, 0)                     # mọi điểm trên viền cùng thành phần với điểm (0, 0)
        regions = 1
        for i in range(n):
            for j in range(n):
                ch = grid[i][j]
                if ch == " ":
                    continue
                if ch == "\\":
                    a, b = i * m + j, (i + 1) * m + j + 1
                else:
                    a, b = i * m + j + 1, (i + 1) * m + j
                if not union(a, b):
                    regions += 1
        return regions

sol_points = Solution().regionsBySlashes
assert [sol_points(list(g)) for g, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n^2 * alpha)` thời gian, `O((n + 1)^2)` bộ nhớ (ít hơn `4 n^2` của Cách 2).

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 1: `[" /", "/ "]`

In [13]:
def trace_dsu(grid):
    n = len(grid)
    dsu = DSU(4 * n * n)
    names = ["trên", "phải", "dưới", "trái"]
    def tag(x):
        cell, k = divmod(x, 4)
        return f"({cell // n},{cell % n}).{names[k]}"
    def u(a, b, why):
        ok = dsu.union(a, b)
        print(f"  gộp {tag(a)} - {tag(b)} ({why}): {'thành công' if ok else 'đã cùng thành phần'}, còn {dsu.components} thành phần")
    print("ban đầu:", dsu.components, "thành phần")
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            ch = grid[i][j]
            print(f"ô ({i},{j}) = {ch!r}")
            if ch == " ":
                u(b, b + 1, "ô trống"); u(b + 1, b + 2, "ô trống"); u(b + 2, b + 3, "ô trống")
            elif ch == "/":
                u(b, b + 3, "dấu /"); u(b + 1, b + 2, "dấu /")
            else:
                u(b, b + 1, "dấu \\"); u(b + 2, b + 3, "dấu \\")
            if i + 1 < n:
                u(b + 2, 4 * ((i + 1) * n + j), "cạnh với ô dưới")
            if j + 1 < n:
                u(b + 1, 4 * (i * n + j + 1) + 3, "cạnh với ô phải")
    return dsu.components

assert trace_dsu([" /", "/ "]) == 2

ban đầu: 16 thành phần
ô (0,0) = ' '
  gộp (0,0).trên - (0,0).phải (ô trống): thành công, còn 15 thành phần
  gộp (0,0).phải - (0,0).dưới (ô trống): thành công, còn 14 thành phần
  gộp (0,0).dưới - (0,0).trái (ô trống): thành công, còn 13 thành phần
  gộp (0,0).dưới - (1,0).trên (cạnh với ô dưới): thành công, còn 12 thành phần
  gộp (0,0).phải - (0,1).trái (cạnh với ô phải): thành công, còn 11 thành phần
ô (0,1) = '/'
  gộp (0,1).trên - (0,1).trái (dấu /): thành công, còn 10 thành phần
  gộp (0,1).phải - (0,1).dưới (dấu /): thành công, còn 9 thành phần
  gộp (0,1).dưới - (1,1).trên (cạnh với ô dưới): thành công, còn 8 thành phần
ô (1,0) = '/'
  gộp (1,0).trên - (1,0).trái (dấu /): thành công, còn 7 thành phần
  gộp (1,0).phải - (1,0).dưới (dấu /): thành công, còn 6 thành phần
  gộp (1,0).phải - (1,1).trái (cạnh với ô phải): thành công, còn 5 thành phần
ô (1,1) = ' '
  gộp (1,1).trên - (1,1).phải (ô trống): thành công, còn 4 thành phần
  gộp (1,1).phải - (1,1).dưới (ô trống): thành công

### 8.2. Cách 4 (điểm lưới) cho ví dụ 3: `["/\\", "\\/"]`

Điểm lưới `3 x 3`, viền gồm `8` điểm đã nối; tâm `(1, 1)` chưa nối. Bốn dấu là bốn cạnh nối các điểm giữa cạnh (không phải góc) với tâm: `/` ở `(0,0)` nối `(0,1)` và `(1,0)`: hai điểm viền đã cùng thành phần ⇒ chu trình, `regions = 2`; `\` ở `(0,1)` nối `(0,1)` và `(1,2)`: cùng viền ⇒ `regions = 3`; `\` ở `(1,0)` nối `(1,0)` và `(2,1)` ⇒ `4`; `/` ở `(1,1)` nối `(1,2)` và `(2,1)` ⇒ `5`.

In [14]:
def trace_points(grid):
    n = len(grid)
    m = n + 1
    dsu = DSU(m * m)
    pid = lambda r, c: r * m + c
    border = [pid(0, c) for c in range(m)] + [pid(m - 1, c) for c in range(m)] + [pid(r, 0) for r in range(m)] + [pid(r, m - 1) for r in range(m)]
    for a, b in zip(border, border[1:]):
        dsu.union(a, b)
    regions = 1
    for i in range(n):
        for j in range(n):
            ch = grid[i][j]
            if ch == " ":
                continue
            (r1, c1), (r2, c2) = ((i, j), (i + 1, j + 1)) if ch == "\\" else ((i, j + 1), (i + 1, j))
            merged = dsu.union(pid(r1, c1), pid(r2, c2))
            if not merged:
                regions += 1
            print(f"dấu {ch!r} ở ô ({i},{j}): nối điểm ({r1},{c1}) - ({r2},{c2}): {'gộp thành phần' if merged else 'khép chu trình -> thêm vùng'}; vùng = {regions}")
    return regions

assert trace_points(["/\\", "\\/"]) == 5

dấu '/' ở ô (0,0): nối điểm (0,1) - (1,0): khép chu trình -> thêm vùng; vùng = 2
dấu '\\' ở ô (0,1): nối điểm (0,1) - (1,2): khép chu trình -> thêm vùng; vùng = 3
dấu '\\' ở ô (1,0): nối điểm (1,0) - (2,1): khép chu trình -> thêm vùng; vùng = 4
dấu '/' ở ô (1,1): nối điểm (1,2) - (2,1): khép chu trình -> thêm vùng; vùng = 5


### 8.3. Cách 3 (DFS) cho ví dụ 2: `[" /", "  "]`

Ba ô trống gộp mỗi ô thành 1 mảnh; ô `/` gộp thành 2 mảnh; các cạnh giữa ô nối tất cả lại: `1` vùng.

In [15]:
assert regions_dfs([" /", "  "]) == 1 == regions_dsu([" /", "  "]) == regions_points([" /", "  "]) == regions_upscale([" /", "  "])
print("ví dụ 2: các cách đều ra 1")

ví dụ 2: các cách đều ra 1


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (Union Find trên 4 tam giác mỗi ô): trực quan, `O(n^2 alpha)`.

```python
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        n = len(grid)
        parent = list(range(4 * n * n))
        count = 4 * n * n

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a: int, b: int) -> None:
            nonlocal count
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                count -= 1

        for i in range(n):
            for j in range(n):
                b = 4 * (i * n + j)
                ch = grid[i][j]
                if ch == " ":
                    union(b, b + 1); union(b + 1, b + 2); union(b + 2, b + 3)
                elif ch == "/":
                    union(b, b + 3); union(b + 1, b + 2)
                else:
                    union(b, b + 1); union(b + 2, b + 3)
                if i + 1 < n:
                    union(b + 2, 4 * ((i + 1) * n + j))
                if j + 1 < n:
                    union(b + 1, 4 * (i * n + j + 1) + 3)
        return count
```

In [16]:
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        n = len(grid)
        parent = list(range(4 * n * n))
        count = 4 * n * n

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a: int, b: int) -> None:
            nonlocal count
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                count -= 1

        for i in range(n):
            for j in range(n):
                b = 4 * (i * n + j)
                ch = grid[i][j]
                if ch == " ":
                    union(b, b + 1); union(b + 1, b + 2); union(b + 2, b + 3)
                elif ch == "/":
                    union(b, b + 3); union(b + 1, b + 2)
                else:
                    union(b, b + 1); union(b + 2, b + 3)
                if i + 1 < n:
                    union(b + 2, 4 * ((i + 1) * n + j))
                if j + 1 < n:
                    union(b + 1, 4 * (i * n + j + 1) + 3)
        return count

sol = Solution()
for g, expected in examples:
    got = sol.regionsBySlashes(list(g))
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[' /', '/ '] -> 2 OK
[' /', '  '] -> 1 OK
['/\\', '\\/'] -> 5 OK


## 10. Vì sao đúng

**(a) Mô hình 4 tam giác.** Mỗi ô là hình vuông; hai đường chéo của nó cắt nó thành 4 tam giác. Một dấu `/` hoặc `\` là một **đường chéo**, nên nó chia ô thành hai nửa mà mỗi nửa là hợp của hai tam giác liền kề: `/` ngăn cách {trên, trái} khỏi {dưới, phải}; `\` ngăn cách {trên, phải} khỏi {dưới, trái}. Ô trống không có đường ngăn. Hai ô kề nhau chia sẻ **một cạnh** hoàn toàn thông (không có dấu nào nằm trên cạnh ô), nên tam giác sát cạnh của hai ô thông nhau. Hai điểm của lưới thuộc cùng một vùng của hình vẽ ⇔ có đường đi liên tục không cắt qua dấu nào ⇔ các tam giác chứa chúng nằm cùng thành phần của đồ thị trên. Vậy **số vùng = số thành phần liên thông** (Cách 2, 3).

**(b) Cách 4 (Euler).** Xét đồ thị phẳng `G` với đỉnh là các điểm lưới và cạnh gồm khung viền và các dấu gạch chéo. Các cạnh chỉ giao nhau tại điểm lưới (mỗi ô một dấu, các dấu ở ô khác nhau chỉ chạm nhau ở góc). Theo công thức Euler cho đồ thị phẳng liên thông hoặc không: số mặt trong khung `F_in = E - V + C` (với `E, V, C` là số cạnh, đỉnh, thành phần của đồ thị gồm khung và các dấu). Thêm từng cạnh: nối hai thành phần khác nhau thì `E` tăng `1` và `C` giảm `1` nên `E - V + C` không đổi; nối hai điểm cùng thành phần thì `E` tăng `1` và `C` giữ nguyên nên `F_in` tăng `1`. Với khung viền ban đầu (một vòng) `F_in = 1`. Do đó `regions = 1 + số cạnh khép chu trình`.

**(c) Phóng to `3 x 3` (Cách 1).** Dấu `/` hoặc `\` trở thành chuỗi ba điểm chéo `(0,2),(1,1),(2,0)` hoặc `(0,0),(1,1),(2,2)`; loang 4 hướng không đi chéo qua chuỗi này (hai điểm chéo kề không chia sẻ cạnh), nên nó chặn đúng như đường chéo thật. Mỗi vùng thật chứa ít nhất một điểm ảnh không tô (tam giác góc nhỏ nhất vẫn chứa góc `3 x 3` tương ứng), và hai điểm ảnh không tô cùng vùng ⇔ nối được bằng loang 4 hướng; do đó số thành phần bằng số vùng.

**Kiểm chứng bằng máy.** Cách 1 (hình học, phóng to) là oracle độc lập với mô hình tam giác; so mọi cách với nó trên toàn bộ lưới `n <= 3` (liệt kê mọi lưới), lưới ngẫu nhiên tới `n = 30` và lưới có cấu trúc. Thêm kiểm tra **tổng diện tích**: tổng số tam giác trong các thành phần bằng `4 n^2`.

In [17]:
from itertools import product

def all_grids(n):
    for cells in product(" /\\", repeat=n * n):
        yield ["".join(cells[i * n:(i + 1) * n]) for i in range(n)]

count = 0
for n in (1, 2, 3):
    for g in all_grids(n):
        ref = regions_upscale(g)
        assert regions_dsu(g) == regions_dfs(g) == regions_points(g) == ref, g
        count += 1
print(f"mọi lưới n <= 3 ({count} lưới): 4 cách khớp nhau")

mọi lưới n <= 3 (19767 lưới): 4 cách khớp nhau


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Phóng to `3 x 3` | `O(9 n^2)` | `O(9 n^2)` |
| 2. Union Find 4 tam giác | `O(n^2 alpha)` | `O(4 n^2)` |
| 3. DFS 4 tam giác | `O(n^2)` | `O(4 n^2)` |
| 4. Union Find điểm lưới | `O(n^2 alpha)` | `O((n + 1)^2)` |

Đo thử với `n = 30` (cỡ tối đa):

In [18]:
rng = random.Random(11)
tests = [random_grid(rng, 30, p) for p in (0.1, 0.3, 0.5, 0.8, 1.0) for _ in range(10)]
ref = [regions_upscale(g) for g in tests]
for name, fn in (("Cách 2 DSU 4 tam giác", sol_dsu), ("Cách 3 DFS", sol_dfs), ("Cách 4 điểm lưới", sol_points), ("nộp", Solution().regionsBySlashes)):
    t0 = time.perf_counter()
    res = [fn(list(g)) for g in tests]
    dt = (time.perf_counter() - t0) * 1000
    assert res == ref, name
    print(f"{name:22} {len(tests)} lưới 30x30: {dt:8.1f} ms")
t0 = time.perf_counter()
[regions_upscale(g) for g in tests]
print(f"{'Cách 1 phóng to':22} {len(tests)} lưới 30x30: {1000 * (time.perf_counter() - t0):8.1f} ms")

Cách 2 DSU 4 tam giác  50 lưới 30x30:     97.5 ms
Cách 3 DFS             50 lưới 30x30:     98.4 ms
Cách 4 điểm lưới       50 lưới 30x30:     19.3 ms


nộp                    50 lưới 30x30:     97.5 ms


Cách 1 phóng to        50 lưới 30x30:    293.7 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm trên mọi lưới nhỏ), đối chiếu mọi cách với oracle phóng to `3 x 3`, liệt kê mọi lưới `n <= 3`, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [19]:
def solutions_list():
    return {
        "dsu": regions_dsu,
        "dfs": regions_dfs,
        "points": regions_points,
        "sol_dsu": sol_dsu,
        "sol_dfs": sol_dfs,
        "sol_points": sol_points,
        "final": Solution().regionsBySlashes,
    }

def run_all(grid, expected=None):
    ref = regions_upscale(grid)
    for name, fn in solutions_list().items():
        assert fn(list(grid)) == ref, (name, grid)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", grid, expected, ref)
    return ref

# 12.1 ví dụ đề
for g, expected in examples:
    assert run_all(g, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([" "], 1),                                  # một ô trống
    (["/"], 2),                                  # một dấu chia đôi
    (["\\"], 2),
    (["  ", "  "], 1),                           # toàn trống
    (["//", "//"], 4),                           # bốn dấu / song song
    (["\\\\", "\\\\"], 4),                       # bốn dấu \
    (["/\\", "\\/"], 5),                         # hình thoi và bốn góc
    (["\\/", "/\\"], 4),                         # hai đường chéo của cả hình vuông cắt nhau ở tâm: dạng X, 4 vùng
    ([" /", "/ "], 2),
    (["/ ", " /"], None),                        # kiểm bằng oracle
    (["/\\ ", "\\/ ", "   "], 3),                # hình thoi ở góc trên trái, phần ngoài thông nhau
    (["   ", " / ", "   "], 1),                  # một dấu ở giữa không chia vùng (đường không tới biên)
    (["///", "///", "///"], 6),                  # năm đường chéo song song (x + y = 1..5) chia thành 6 dải
]
for g, expected in edge:
    run_all(g, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên mọi lưới nhỏ, xác nhận lời giải đúng không bị lỗi)
def regions_generic(grid, within, between=True):
    """Union Find 4 tam giác với 'within' tùy biến (để mô phỏng lỗi)."""
    n = len(grid)
    dsu = DSU(4 * n * n)
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            for a, c in within(grid[i][j]):
                dsu.union(b + a, b + c)
            if between:
                if i + 1 < n:
                    dsu.union(b + 2, 4 * ((i + 1) * n + j))
                if j + 1 < n:
                    dsu.union(b + 1, 4 * (i * n + j + 1) + 3)
    return dsu.components

correct_within = lambda ch: [(0, 1), (1, 2), (2, 3)] if ch == " " else ([(0, 3), (1, 2)] if ch == "/" else [(0, 1), (2, 3)])
swapped_within = lambda ch: [(0, 1), (1, 2), (2, 3)] if ch == " " else ([(0, 1), (2, 3)] if ch == "/" else [(0, 3), (1, 2)])
blank_separate = lambda ch: [] if ch == " " else correct_within(ch)

def wrong_no_between(grid):                           # chỉ gộp trong ô, quên gộp giữa các ô kề
    return regions_generic(grid, correct_within, between=False)

def wrong_swapped(grid):                              # đổi vai trò '/' và '\'
    return regions_generic(grid, swapped_within)

def wrong_blank_separate(grid):                       # ô trống bị coi là bốn mảnh rời
    return regions_generic(grid, blank_separate)

def wrong_points_no_border(grid):                     # Cách 4 nhưng quên nối các điểm trên viền
    n = len(grid)
    m = n + 1
    dsu = DSU(m * m)
    pid = lambda r, c: r * m + c
    regions = 1
    for i in range(n):
        for j in range(n):
            ch = grid[i][j]
            if ch == " ":
                continue
            a, b = (pid(i, j), pid(i + 1, j + 1)) if ch == "\\" else (pid(i, j + 1), pid(i + 1, j))
            if not dsu.union(a, b):
                regions += 1
    return regions

def find_counter(wrong, max_n=3):
    for n in range(1, max_n + 1):
        for g in all_grids(n):
            if wrong(g) != regions_upscale(g):
                return g
    return None

counters = {}
for name, fn in (("quên gộp giữa các ô", wrong_no_between), ("đổi vai trò / và \\", wrong_swapped),
                 ("ô trống là bốn mảnh rời", wrong_blank_separate), ("điểm lưới quên nối viền", wrong_points_no_border)):
    g = find_counter(fn)
    assert g is not None, name
    counters[name] = g
    print(f"{name:28} grid={g}: sai -> {fn(g)}, đúng -> {regions_upscale(g)}")
    run_all(g)
print("12.3 phản ví dụ: OK")

quên gộp giữa các ô          grid=['  ', '  ']: sai -> 4, đúng -> 1
đổi vai trò / và \           grid=['  ', ' /']: sai -> 1, đúng -> 2
ô trống là bốn mảnh rời      grid=[' ']: sai -> 4, đúng -> 1
điểm lưới quên nối viền      grid=['/']: sai -> 1, đúng -> 2
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ-trung bình: n = 4..12, nhiều mật độ dấu
rng = random.Random(4959)
for _ in range(600):
    n = rng.randint(4, 12)
    run_all(random_grid(rng, n, rng.choice([0.1, 0.3, 0.5, 0.8, 1.0])))
print("12.4 ngẫu nhiên n = 4..12 (600 lưới): OK")

# 12.5 ngẫu nhiên lớn: n = 13..30
rng = random.Random(5959)
for _ in range(150):
    n = rng.randint(13, 30)
    run_all(random_grid(rng, n, rng.choice([0.1, 0.3, 0.5, 0.8, 1.0])))
print("12.5 ngẫu nhiên n = 13..30 (150 lưới): OK")

# 12.6 các lưới có cấu trúc n = 30: bàn cờ, kim cương lặp, toàn / và toàn \, hình xoắn
n = 30
special = {
    "toàn trống": [" " * n] * n,
    "toàn /": ["/" * n] * n,
    "toàn \\": ["\\" * n] * n,
    "bàn cờ / \\": [("/\\" * n)[:n] if i % 2 == 0 else ("\\/" * n)[:n] for i in range(n)],
    "hàng đầu đầy /": ["/" * n] + [" " * n] * (n - 1),
    "đường chéo chính \\": ["".join("\\" if i == j else " " for j in range(n)) for i in range(n)],
}
for name, g in special.items():
    ref = regions_upscale(g)
    for fname, fn in solutions_list().items():
        assert fn(list(g)) == ref, (name, fname)
    print(f"  {name:22} -> {ref} vùng")
assert regions_dsu(special["toàn trống"]) == 1
assert regions_dsu(special["toàn /"]) == 2 * n
assert regions_dsu(special["đường chéo chính \\"]) == 2
print("12.6 lưới có cấu trúc n = 30: OK")

# 12.7 tính chất: số vùng >= 1; đảo gương (đổi / <-> \ và đảo mỗi hàng) không đổi số vùng; tổng kích thước thành phần = 4 n^2
rng = random.Random(7959)
def mirror(grid):
    flip = {" ": " ", "/": "\\", "\\": "/"}
    return ["".join(flip[ch] for ch in reversed(row)) for row in grid]

for _ in range(200):
    n = rng.randint(1, 12)
    g = random_grid(rng, n, rng.choice([0.2, 0.6, 1.0]))
    r = Solution().regionsBySlashes(list(g))
    assert r >= 1
    assert Solution().regionsBySlashes(mirror(g)) == r            # đối xứng gương trái-phải
    # tổng kích thước các thành phần của DSU = 4 n^2
    dsu = DSU(4 * n * n)
    sizes = {}
    for x in range(4 * n * n):
        sizes[dsu.find(x)] = sizes.get(dsu.find(x), 0) + 1
    assert sum(sizes.values()) == 4 * n * n
print("12.7 tính chất (>= 1, đối xứng gương): OK")

12.4 ngẫu nhiên n = 4..12 (600 lưới): OK


12.5 ngẫu nhiên n = 13..30 (150 lưới): OK
  toàn trống             -> 1 vùng
  toàn /                 -> 60 vùng
  toàn \                 -> 60 vùng
  bàn cờ / \             -> 481 vùng
  hàng đầu đầy /         -> 2 vùng
  đường chéo chính \     -> 2 vùng
12.6 lưới có cấu trúc n = 30: OK
12.7 tính chất (>= 1, đối xứng gương): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên gộp các tam giác giữa hai ô kề

Chỉ gộp trong từng ô sẽ coi mỗi ô là một tập mảnh riêng, tạo quá nhiều vùng.

In [22]:
g = counters["quên gộp giữa các ô"]
print("grid =", g, "| sai ->", wrong_no_between(g), "| đúng ->", Solution().regionsBySlashes(list(g)))
assert wrong_no_between(g) != Solution().regionsBySlashes(list(g))
print("ví dụ 2 [' /', '  ']:", wrong_no_between([" /", "  "]), "(sai) so với 1")

grid = ['  ', '  '] | sai -> 4 | đúng -> 1
ví dụ 2 [' /', '  ']: 5 (sai) so với 1


Vì sao sai: hai ô kề chung một cạnh không có dấu chặn, nên các tam giác hai bên thông nhau. **Đúng:** gộp `dưới(i, j)` với `trên(i + 1, j)` và `phải(i, j)` với `trái(i, j + 1)`.

### Lỗi 2 — Đổi vai trò của `/` và `\`

Gộp `0-1`/`2-3` cho `/` (đáng ra là cho `\`) cho kết quả sai ở nhiều lưới.

In [23]:
g = counters["đổi vai trò / và \\"]
print("grid =", g, "| sai ->", wrong_swapped(g), "| đúng ->", Solution().regionsBySlashes(list(g)))
assert wrong_swapped(g) != Solution().regionsBySlashes(list(g))

grid = ['  ', ' /'] | sai -> 1 | đúng -> 2


Vì sao sai: `/` nối góc dưới trái với góc trên phải, tách {trên, trái} khỏi {dưới, phải}; `\` tách {trên, phải} khỏi {dưới, trái}. **Đúng:** `/` -> gộp `0-3` và `1-2`; `\` -> gộp `0-1` và `2-3`.

### Lỗi 3 — Coi ô trống là bốn mảnh rời

Ô trống không có đường ngăn nên cả bốn tam giác thông nhau.

In [24]:
g = counters["ô trống là bốn mảnh rời"]
print("grid =", g, "| sai ->", wrong_blank_separate(g), "| đúng ->", Solution().regionsBySlashes(list(g)))
assert wrong_blank_separate(g) != Solution().regionsBySlashes(list(g))

grid = [' '] | sai -> 4 | đúng -> 1


Vì sao sai: bốn tam giác của ô trống phải được gộp (ba phép gộp). **Đúng:** `union(b, b+1)`, `union(b+1, b+2)`, `union(b+2, b+3)`.

### Lỗi 4 — Ở cách điểm lưới, quên nối các điểm trên viền

Khung ngoài cùng cũng là đường chắn: các điểm viền phải cùng một thành phần ngay từ đầu.

In [25]:
g = counters["điểm lưới quên nối viền"]
print("grid =", g, "| sai ->", wrong_points_no_border(g), "| đúng ->", Solution().regionsBySlashes(list(g)))
assert wrong_points_no_border(g) != Solution().regionsBySlashes(list(g))

grid = ['/'] | sai -> 1 | đúng -> 2


Vì sao sai: một dấu nối hai điểm viền khép chu trình cùng khung (tách một vùng) nhưng nếu các điểm viền chưa được nối thì dấu đó bị coi là một phép gộp bình thường. **Đúng:** nối mọi điểm viền trước.

### Lỗi 5 — Nhầm ký tự gạch chéo ngược

Trong Python, `'\'` gây lỗi cú pháp; phải viết `'\\'`. Khi so sánh `grid[i][j] == "\\"` mới đúng (một ký tự).

In [26]:
print(len("\\"), "\\" == chr(92), ["/\\", "\\/"][0][1] == "\\")

1 True True


Vì sao: ký tự thoát `\` đứng một mình nên cần `\\`. **Đúng:** dùng `"\\"` hoặc `chr(92)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm vùng bị chia bởi các đường chéo trong lưới ô" ⇒ **tách ô thành 4 tam giác** (hoặc phóng to `3 x 3`) và **đếm thành phần liên thông**.
- **Chỉ số phẳng** `4 * (i * n + j) + k` với `k` là trên, phải, dưới, trái; `/` gộp `0-3`, `1-2`; `\` gộp `0-1`, `2-3`.
- **Cách điểm lưới:** mỗi dấu là một cạnh; cạnh khép chu trình thì tăng số vùng (công thức Euler).
- Dùng Union Find khi muốn đếm thành phần mà không cần duyệt; `count` giảm khi gộp thành công.
- Cùng họ: Number of Islands (200), Number of Provinces (547), Surrounded Regions (130), Making A Large Island (827), Redundant Connection (684).
- Test nên có: `n = 1` với ba ký tự, lưới toàn trống, hình thoi (ví dụ 3), đường chéo xuyên suốt, `n = 30`.

## 15. Biến thể

1. **Lưới chữ nhật `n x m`** (không nhất thiết vuông): chỉ đổi chỉ số.
2. **Diện tích từng vùng** (bài tập 5): đếm số tam giác (một phần tư ô) trong mỗi thành phần; tổng bằng `4 n^2`.
3. **Thêm ký hiệu khác** (ví dụ `'|'` hay `'-'`): thay bảng gộp trong ô.
4. Bài liên quan: Number of Islands (200).

Dưới đây là biến thể 2 (diện tích từng vùng theo đơn vị một phần tư ô) kiểm bằng DFS độc lập.

In [27]:
def region_areas(grid: List[str]) -> List[int]:
    n = len(grid)
    dsu = DSU(4 * n * n)
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            for a, c in correct_within(grid[i][j]):
                dsu.union(b + a, b + c)
            if i + 1 < n:
                dsu.union(b + 2, 4 * ((i + 1) * n + j))
            if j + 1 < n:
                dsu.union(b + 1, 4 * (i * n + j + 1) + 3)
    sizes = {}
    for x in range(4 * n * n):
        r = dsu.find(x)
        sizes[r] = sizes.get(r, 0) + 1
    return sorted(sizes.values())

def region_areas_dfs(grid: List[str]) -> List[int]:
    n = len(grid)
    adj = [[] for _ in range(4 * n * n)]
    def link(a, c):
        adj[a].append(c); adj[c].append(a)
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            for a, c in correct_within(grid[i][j]):
                link(b + a, b + c)
            if i + 1 < n:
                link(b + 2, 4 * ((i + 1) * n + j))
            if j + 1 < n:
                link(b + 1, 4 * (i * n + j + 1) + 3)
    seen = [False] * (4 * n * n)
    out = []
    for s in range(4 * n * n):
        if seen[s]:
            continue
        seen[s] = True
        stack, size = [s], 0
        while stack:
            x = stack.pop()
            size += 1
            for y in adj[x]:
                if not seen[y]:
                    seen[y] = True
                    stack.append(y)
        out.append(size)
    return sorted(out)

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 8)
    g = random_grid(rng, n, rng.choice([0.2, 0.6, 1.0]))
    a = region_areas(g)
    assert a == region_areas_dfs(g) and sum(a) == 4 * n * n and len(a) == regions_upscale(g)
assert region_areas(["/\\", "\\/"]) == [2, 2, 2, 2, 8]
print("diện tích vùng của ví dụ 3:", region_areas(["/\\", "\\/"]), "(bốn góc mỗi góc 2 tam giác, hình thoi 8 tam giác; tổng 16)")

diện tích vùng của ví dụ 3: [2, 2, 2, 2, 8] (bốn góc mỗi góc 2 tam giác, hình thoi 8 tam giác; tổng 16)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Phóng to `3 x 3` | Điểm ảnh + tô loang | `O(9 n^2)` | `O(9 n^2)` | Oracle hình học |
| 2. Union Find 4 tam giác | Gộp mảnh trong/giữa ô | `O(n^2 alpha)` | `O(4 n^2)` | **Cách nộp chính** |
| 3. DFS 4 tam giác | Thành phần liên thông | `O(n^2)` | `O(4 n^2)` | Không cần Union Find |
| 4. Union Find điểm lưới | Đếm chu trình (Euler) | `O(n^2 alpha)` | `O((n + 1)^2)` | Ít bộ nhớ nhất |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(piece(2, 1, 0, TOP), piece(2, 0, 0, BOTTOM))
print(len("/\\"), "/\\"[1] == "\\")
d = DSU(4); print(d.union(0, 1), d.union(1, 0), d.components)
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `grid = ["//", "//"]` (bốn dấu `/`): ghi số vùng.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành bước gộp trong ô cho Cách 2.

In [30]:
def exercise_3_dsu(grid: List[str]) -> int:
    n = len(grid)
    dsu = DSU(4 * n * n)
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            ch = grid[i][j]
            # TODO 1: gộp các tam giác trong ô theo ký tự ch (ô trống / '/' / '\\')
            # TODO 2: gộp giữa các ô: dưới của (i, j) với trên của (i + 1, j); phải của (i, j) với trái của (i, j + 1)
    return dsu.components


# for g, expected in examples:
#     assert exercise_3_dsu(g) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_points(grid)` bằng Union Find trên điểm lưới (đếm chu trình).

In [31]:
def exercise_4_points(grid: List[str]) -> int:
    # TODO
    pass


# for g, expected in examples:
#     assert exercise_4_points(g) == expected

### Bài 5 — Mini challenge: diện tích từng vùng (khó)

Viết `exercise_5_areas(grid)` trả về danh sách **diện tích** (đếm theo số tam giác, tức một phần tư ô) của mỗi vùng, **sắp xếp tăng dần**.

```python
exercise_5_areas([" /", "  "])      -> [16]                  # một vùng gồm cả 16 tam giác
exercise_5_areas(["/\\", "\\/"])    -> [2, 2, 2, 2, 8]       # bốn góc (mỗi góc 2 tam giác) và hình thoi (8 tam giác)
```
Gợi ý: Union Find trên 4 tam giác rồi đếm kích thước từng thành phần.

In [32]:
def exercise_5_areas(grid: List[str]) -> List[int]:
    # TODO
    pass


# assert exercise_5_areas([" /", "  "]) == [16]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `piece(n, i, j, k) = 4 * (i * n + j) + k`; `TOP = 0`, `BOTTOM = 2`.
- Mức 2: `piece(2, 1, 0, 0) = 4 * 2 = 8`; `piece(2, 0, 0, 2) = 2`.
- Mức 3: `8 2`, `2 True`, `True False 3`.

**Bài 2**
- Mức 1: các dấu `/` ở bốn ô nối thành các đường chéo song song.
- Mức 2: ba đường chéo song song (`x + y = 1, 2, 3`) chia hình vuông thành các dải.
- Mức 3: `4`.

**Bài 3**
- Mức 1: ô trống gộp `b-b+1`, `b+1-b+2`, `b+2-b+3`.
- Mức 2: `/` gộp `b-b+3` và `b+1-b+2`; `\` gộp `b-b+1` và `b+2-b+3`.
- Mức 3: giữa ô: `b+2` với `4*((i+1)*n+j)` và `b+1` với `4*(i*n+j+1)+3`.

**Bài 4**
- Mức 1: điểm lưới `(n + 1) x (n + 1)`; nối mọi điểm viền.
- Mức 2: `\` nối `(i, j)`-`(i+1, j+1)`; `/` nối `(i, j+1)`-`(i+1, j)`.
- Mức 3: `regions = 1`; mỗi cạnh nối hai điểm đã cùng thành phần thì `regions += 1`.

**Bài 5**
- Mức 1: dùng `DSU` trên `4 n^2` tam giác.
- Mức 2: đếm kích thước từng gốc `find(x)`.
- Mức 3: `sorted(sizes.values())`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(piece(2, 1, 0, TOP), piece(2, 0, 0, BOTTOM))
print(len("/\\"), "/\\"[1] == "\\")
d = DSU(4); print(d.union(0, 1), d.union(1, 0), d.components)

assert (piece(2, 1, 0, TOP), piece(2, 0, 0, BOTTOM)) == (8, 2)
assert (len("/\\"), "/\\"[1] == "\\") == (2, True)
d = DSU(4)
assert (d.union(0, 1), d.union(1, 0), d.components) == (True, False, 3)

8 2
2 True
True False 3


### Đáp án Bài 2

Bốn dấu `/` chia hình vuông thành `4` vùng.

In [34]:
hand_answer_solution = 4
assert Solution().regionsBySlashes(["//", "//"]) == hand_answer_solution == regions_upscale(["//", "//"])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_dsu_solution(grid: List[str]) -> int:
    n = len(grid)
    dsu = DSU(4 * n * n)
    for i in range(n):
        for j in range(n):
            b = 4 * (i * n + j)
            ch = grid[i][j]
            if ch == " ":
                dsu.union(b, b + 1); dsu.union(b + 1, b + 2); dsu.union(b + 2, b + 3)
            elif ch == "/":
                dsu.union(b, b + 3); dsu.union(b + 1, b + 2)
            else:
                dsu.union(b, b + 1); dsu.union(b + 2, b + 3)
            if i + 1 < n:
                dsu.union(b + 2, 4 * ((i + 1) * n + j))
            if j + 1 < n:
                dsu.union(b + 1, 4 * (i * n + j + 1) + 3)
    return dsu.components

for g, expected in examples:
    assert exercise_3_dsu_solution(g) == expected
rng = random.Random(3)
for _ in range(200):
    g = random_grid(rng, rng.randint(1, 10), rng.choice([0.3, 0.7, 1.0]))
    assert exercise_3_dsu_solution(g) == regions_upscale(g)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_points_solution(grid: List[str]) -> int:
    n = len(grid)
    m = n + 1
    dsu = DSU(m * m)
    for r in range(m):
        for c in range(m):
            if r in (0, m - 1) or c in (0, m - 1):
                dsu.union(r * m + c, 0)
    regions = 1
    for i in range(n):
        for j in range(n):
            ch = grid[i][j]
            if ch == " ":
                continue
            a, b = ((i * m + j, (i + 1) * m + j + 1) if ch == "\\" else (i * m + j + 1, (i + 1) * m + j))
            if not dsu.union(a, b):
                regions += 1
    return regions

for g, expected in examples:
    assert exercise_4_points_solution(g) == expected
rng = random.Random(4)
for _ in range(200):
    g = random_grid(rng, rng.randint(1, 10), rng.choice([0.3, 0.7, 1.0]))
    assert exercise_4_points_solution(g) == regions_upscale(g)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_areas_solution(grid: List[str]) -> List[int]:
    return region_areas(grid)

assert exercise_5_areas_solution([" /", "  "]) == [16]
assert exercise_5_areas_solution(["/\\", "\\/"]) == [2, 2, 2, 2, 8]
print("diện tích vùng của ví dụ 3:", exercise_5_areas_solution(["/\\", "\\/"]))
assert sum(exercise_5_areas_solution(["/\\", "\\/"])) == 16 and len(exercise_5_areas_solution(["/\\", "\\/"])) == 5
rng = random.Random(5)
for _ in range(200):
    g = random_grid(rng, rng.randint(1, 8), rng.choice([0.3, 0.7, 1.0]))
    a = exercise_5_areas_solution(g)
    assert a == region_areas_dfs(g) and sum(a) == 4 * len(g) ** 2
print("Bài 5 OK")

diện tích vùng của ví dụ 3: [2, 2, 2, 2, 8]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (Union Find trên 4 tam giác mỗi ô)

```python
class Solution:
    def regionsBySlashes(self, grid: List[str]) -> int:
        n = len(grid)
        parent = list(range(4 * n * n))          # 4 tam giác mỗi ô: 0 trên, 1 phải, 2 dưới, 3 trái
        count = 4 * n * n

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            nonlocal count
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                count -= 1

        for i in range(n):
            for j in range(n):
                b = 4 * (i * n + j)
                ch = grid[i][j]
                if ch == " ":                    # ô trống: cả bốn tam giác thông nhau
                    union(b, b + 1); union(b + 1, b + 2); union(b + 2, b + 3)
                elif ch == "/":                  # {trên, trái} | {dưới, phải}
                    union(b, b + 3); union(b + 1, b + 2)
                else:                            # '\\': {trên, phải} | {dưới, trái}
                    union(b, b + 1); union(b + 2, b + 3)
                if i + 1 < n:
                    union(b + 2, 4 * ((i + 1) * n + j))        # dưới (i, j) - trên (i + 1, j)
                if j + 1 < n:
                    union(b + 1, 4 * (i * n + j + 1) + 3)      # phải (i, j) - trái (i, j + 1)
        return count
```

Công thức nhớ nhanh: `/` gộp `0-3`, `1-2`; `\` gộp `0-1`, `2-3`; giữa ô: `2 -> trên của ô dưới`, `1 -> trái của ô phải`; số vùng = `count`.

### Checklist trước khi submit

- [ ] Mỗi ô tách thành **4 tam giác** (không phải 2)?
- [ ] `/` gộp `0-3` và `1-2`; `\` gộp `0-1` và `2-3`; ô trống gộp cả bốn?
- [ ] Có gộp **giữa các ô kề** (dưới-trên, phải-trái)?
- [ ] Ký tự gạch chéo ngược xử lý đúng (`"\\"` trong mã Python)?
- [ ] Đã thử: `n = 1` với ba ký tự, lưới toàn trống, hình thoi (ví dụ 3), `n = 30`?
- [ ] Tên method đúng `regionsBySlashes(self, grid)` và trả `int`?